# 2-stage adaptation / 1-stage E2E comparison

ERM ResNet checkpointからbase backbone・共有classifierを固定して始める2-stage adaptationと、ImageNet初期化から更新する1-stage E2Eを比べる。比較は `fc` / `stage4+fc` と GroupDRO step size の組ごとに行う。各条件は seed 42 の1本なので、差は探索的な観察としてのみ扱う。

## 1. 入力と比較対

`runs/` に取り込んだ iterative run を config から発見する。各変調範囲 × step size に、ERM checkpointを親にするfrozen 2-stage runと、ImageNet初期化のunfrozen E2E runがちょうど1本ずつ無ければ止める。checkpoint は各 run の最後の stage で val AUROC が最大のものを使う。

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rootutils

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)
from analysis.common.paths import split_csv
from analysis.common.predictions import cache_path, load_cache
from analysis.iterative_probe._shared import collect_epoch_rows, read_condition
from analysis.iterative_probe.groups import demographics_of, group_rows, summarize

PACKAGE = ROOT / "analysis" / "iterative_probe"
RUNS = PACKAGE / "runs"
CACHE = PACKAGE / "cache"
RESULTS = PACKAGE / "results"
FIGURES = PACKAGE / "figures"
RESULTS.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

run_dirs = [path for path in sorted(RUNS.iterdir()) if (path / "run.json").is_file()]
PARENT_RUN_ID = "20260921T103036Z-resnet-chexpert-s42-5538"


def training_regime(record: dict) -> str | None:
    checkpoint = str(record["backbone_checkpoint"])
    if record["backbone_frozen"] and PARENT_RUN_ID in checkpoint:
        return "two-stage adaptation"
    if not record["backbone_frozen"] and checkpoint.endswith("resnet50_imagenet.pth"):
        return "one-stage E2E"
    return None


conditions = pd.DataFrame([{"run_id": path.name, **read_condition(path)} for path in run_dirs])
conditions["regime"] = [training_regime(record) for record in conditions.to_dict("records")]
conditions = conditions[conditions["modulation"].isin(["fc", "stage4+fc"])].copy()
conditions = conditions[conditions["step_size"].isin([1e-3, 1e-2])].copy()
conditions = conditions[conditions["regime"].notna()].copy()
counts = conditions.groupby(["modulation", "step_size", "regime"]).size()
expected = pd.MultiIndex.from_product(
    [["fc", "stage4+fc"], [1e-3, 1e-2], ["two-stage adaptation", "one-stage E2E"]],
    names=["modulation", "step_size", "regime"],
)
missing = expected.difference(counts.index)
duplicates = counts[counts != 1]
if len(missing) or not duplicates.empty:
    raise ValueError(f"2-stage / E2E の比較対が不完全: missing={list(missing)}, duplicates={duplicates.to_dict()}")
conditions.sort_values(["modulation", "step_size", "regime"])

## 2. Validation の global・hidden cohort・属性群

hidden cohort は stage ごとに再構成されるため、cohort番号をrun間・stage間で対応付けない。各epochの worst / gap と `q` の集中度を比較する。

In [ ]:
epoch_rows = pd.DataFrame(row for run_id in conditions["run_id"] for row in collect_epoch_rows(RUNS / run_id)).merge(
    conditions[["run_id", "regime"]], on="run_id", validate="many_to_one"
)
q_columns = [column for column in epoch_rows if column.startswith("train/group_dro/q_")]
stage_rows = epoch_rows[epoch_rows["stage"].str.startswith("stage")].copy()
q = stage_rows[q_columns].to_numpy(dtype=float)
stage_rows["q_max"] = np.nanmax(q, axis=1)
stage_rows["q_entropy"] = -(np.where(q > 0, q * np.log(q), 0)).sum(axis=1)
epoch_rows.to_csv(RESULTS / "backbone_freeze_epoch_metrics.csv", index=False)

METRICS = {
    "val/auroc": "global AUROC",
    "val/bacc": "global bACC",
    "val/hidden_min_auroc": "hidden worst AUROC",
    "val/hidden_auroc_gap": "hidden AUROC gap",
    "q_max": "max q",
    "q_entropy": "q entropy",
}
plot_rows = pd.concat([epoch_rows.assign(source="epoch"), stage_rows.assign(source="stage")], ignore_index=True)
figure, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
for axis, (metric, title) in zip(axes.ravel(), METRICS.items()):
    source = "stage" if metric.startswith("q_") else "epoch"
    for (modulation, step_size, regime), part in plot_rows[plot_rows["source"].eq(source)].groupby(
        ["modulation", "step_size", "regime"], sort=True
    ):
        axis.plot(
            part["run_epoch"],
            part[metric],
            color={1e-3: "#4c78a8", 1e-2: "#f58518"}[step_size],
            linestyle="--" if regime == "two-stage adaptation" else "-",
            label=f"{modulation}, {step_size:g}, {regime}",
        )
    axis.set_title(title)
    axis.set_xlabel("run epoch")
    axis.legend(fontsize=8)
figure.tight_layout()
figure.savefig(FIGURES / "backbone_freeze_validation_curves.png", dpi=180)
plt.show()

## 3. Test の属性群・交差群

val全体で感度0.9になる閾値をrunごとに定め、testへ固定して適用する。age・sex・raceの単独群から3属性交差群まで、同一母集団で worst / best / gap とTPR/FPR gapを出す。

In [ ]:
def threshold_at_sensitivity(target: np.ndarray, probability: np.ndarray, sensitivity: float = 0.9) -> float:
    positives = probability[target == 1]
    if len(positives) == 0:
        raise ValueError("val split に陽性例がない")
    return float(np.quantile(positives, 1 - sensitivity))


val_images = pd.read_csv(split_csv("chexpert", "val"))["image"].to_numpy()
test_images = pd.read_csv(split_csv("chexpert", "test"))["image"].to_numpy()
demographics = demographics_of("test")
test_rows = []
for record in conditions.to_dict("records"):
    run_dir = RUNS / record["run_id"]
    val_cache = load_cache(cache_path(CACHE, record["run_id"], "val"), val_images)
    test_cache = load_cache(cache_path(CACHE, record["run_id"], "test"), test_images)
    threshold = threshold_at_sensitivity(val_cache["target"], val_cache["probabilities"][:, 1])
    evaluated_cache = dict(test_cache)
    evaluated_cache["predictions"] = (test_cache["probabilities"][:, 1] >= threshold).astype(int)
    label = f"{record['modulation']} / {record['step_size']:g} / {record['regime']}"
    model = {**record, "label": label, "cache": evaluated_cache}
    for row in group_rows(model, demographics):
        test_rows.append({**record, "threshold": threshold, **row})

test_groups = pd.DataFrame(test_rows)
test_summary = (
    test_groups.groupby(["run_id", "modulation", "step_size", "regime", "grouping"], sort=True)
    .apply(summarize)
    .reset_index()
)
test_groups.to_csv(RESULTS / "backbone_freeze_test_group_metrics.csv", index=False)
test_summary.to_csv(RESULTS / "backbone_freeze_test_group_summary.csv", index=False)
test_summary.sort_values(["grouping", "modulation", "step_size", "regime"])

## 4. 読み取り

結果は `reports/backbone_freeze_comparison.md` に、観察事実・解釈候補・追加確認を分けて記録する。hidden cohortの改善と、test属性群のworst改善を同一視しない。どちらも単一seedでは次の再現候補を選ぶための根拠に限る。